transformer简单实现，best loss 0.451

注意本文的代码是用.py运行，若要在kaggle上运行，请自行修改代码，如读入数据和保存数据的代码

本文分为两版code

code1为看历史特征和标签，用未来特征预测未来标签，用最基础的transformer模型实现，参数设置较为简单
平均loss可以达到0.55-0.60


code2为看历史特征和标签，用未来特征和前一个未来标签（第一个标签为历史最后一天的标签）预测未来标签

code2在模型层数设置较深，超参数设置比较高的情况下，最好loss可以达到0.451，较低参数情况下平均在0.50左右

In [ ]:
#code 1

import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
import math
from transformers import get_cosine_schedule_with_warmup
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler, RobustScaler
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import OneCycleLR
import numpy as np
from tqdm import tqdm
import re

pd.set_option('display.max_columns', None)
root = Path('data')

def data_processing():
    # 读数据
    train = pd.read_csv(root / "train.csv", parse_dates=["date"])
    test = pd.read_csv(root / "test.csv", parse_dates=["date"])
    stores = pd.read_csv(root / "stores.csv")
    oil = pd.read_csv(root / "oil.csv", parse_dates=["date"])
    transactions = pd.read_csv(root / "transactions.csv", parse_dates=["date"])
    holidays = pd.read_csv(root / "holidays_events.csv", parse_dates=["date"])
    sample = pd.read_csv(root / "sample_submission.csv")

    # 预处理train文件
    df = train.merge(stores, on='store_nbr', how='left')
    df = df.merge(oil, on='date', how='left')
    # 填缺+改列名(防止重名)
    df["dcoilwtico"] = df["dcoilwtico"].interpolate(method="linear", limit_direction="both")
    df.rename(columns={"type": "stores_type"}, inplace=True)


    # 预处理holidays文件 -> 处理文件字符串，改列名
    def clean_text(text):
        text = text.lower()
        text = re.sub(r"[^a-z ]", " ", text)  # 保留字母
        return text.strip().split()
    def clean_word(text):
        text_num = re.sub(r"[^1-9 +-]", " ", text).strip()
        return text_num

    holidays["desc_clean"] = holidays["description"].astype(str).map(clean_text)
    holidays["desc_num"] = holidays["description"].astype(str).map(clean_word)
    holidays['transferred'] = holidays['transferred'].astype(str)
    holidays['description'] = holidays.apply(lambda row: row["desc_clean"] + [row["desc_num"]], axis=1)
    holidays['type'] = holidays.apply(lambda row: [row["type"], row["transferred"]], axis=1)
    holidays.drop(columns=["desc_clean", "desc_num", "transferred"], inplace=True)
    holidays = holidays.rename(columns={"type": "day_type"})

    # 编码
    day_type_label = LabelEncoder()
    description_label = LabelEncoder()
    city_label = LabelEncoder()
    state_label = LabelEncoder()
    family_label = LabelEncoder()
    stores_label = LabelEncoder()
    # 简单字符串编码
    df['city_enc'] = city_label.fit_transform(df['city'])
    df['state_enc'] = state_label.fit_transform(df['state'])
    df['family_enc'] = family_label.fit_transform(df['family'])
    df['stores_type'] = stores_label.fit_transform(df['stores_type'])


    # 句子+复杂日期类型编码
    # 预编码
    day_type_all = [y for x in holidays['day_type'] for y in x] + ['Work Day', 'Endweek', 'nan']
    description_all = [y for x in holidays['description'] for y in x] + ['nan']
    day_type_label.fit_transform(day_type_all)
    description_label.fit_transform(description_all)
    # 编码holidays字符串
    holidays['day_type_enc'] = holidays['day_type'].apply(lambda x: day_type_label.transform(x))
    holidays['description_enc'] = holidays['description'].apply(lambda x: description_label.transform(x))

    # holidays处理
    # 按国家，州， 城市活动日期分类
    nat_holidays = holidays[holidays["locale"] == "National"]
    reg_holidays = holidays[holidays["locale"] == "Regional"]
    loc_holidays = holidays[holidays["locale"] == "Local"]

    # 求最大值
    des_maxlen = max(max(nat_holidays['description_enc'].apply(lambda x: len(x) if isinstance(x, np.ndarray) else [])),
                     max(reg_holidays['description_enc'].apply(lambda x: len(x) if isinstance(x, np.ndarray) else [])),
                     max(loc_holidays['description_enc'].apply(lambda x: len(x) if isinstance(x, np.ndarray) else [])))
    day_maxlen = max(max(nat_holidays['day_type_enc'].apply(lambda x: len(x) if isinstance(x, np.ndarray) else [])),
                     max(reg_holidays['day_type_enc'].apply(lambda x: len(x) if isinstance(x, np.ndarray) else [])),
                     max(loc_holidays['day_type_enc'].apply(lambda x: len(x) if isinstance(x, np.ndarray) else [])))

    # 补充编码提取
    description_nan_num = description_label.transform(['nan']).tolist()
    day_type_nan_num = day_type_label.transform(['nan']).tolist()
    workday_num = day_type_label.transform(['Work Day']).item()
    endweek_num = day_type_label.transform(['Endweek']).item()

    def process_holidays(df, des_maxlen, dty_maxlen, description_nan_num, day_type_nan_num):
        # 对需要处理的列执行统一的填充操作
        for col, max_len, pad_value in [('description_enc', des_maxlen, description_nan_num),
                                        ('day_type_enc', dty_maxlen, day_type_nan_num)]:
            df.loc[:, col] = df[col].apply(
                lambda x: x.tolist() + pad_value * (max_len - len(x)) if len(x) < max_len else x.tolist())
        return df

    # 对所有 DataFrame 执行相同的操作
    for holidays_df in [nat_holidays, reg_holidays, loc_holidays]:
        holidays_df = process_holidays(holidays_df, des_maxlen, day_maxlen, description_nan_num, day_type_nan_num)

    def merge_holidays(df, holidays_df, mask_col):
        """
        df: 主表
        holidays_df: 假期表
        on_col: 用来 merge 的列，一般是 'date'
        mask_col: 用来筛选行的列，比如 'state' 或 'city'
        """
        # 合并重复日期
        holidays_df = (
            holidays_df.groupby(["date", "locale_name"], as_index=False).agg({
                "day_type_enc": lambda x: sum(x, []),
                "description_enc": lambda x: sum(x, []),
            })
        )

        # merge 带新信息
        df = df.merge(
            holidays_df[['date', 'day_type_enc', 'description_enc', 'locale_name']],
            left_on=['date', mask_col],
            right_on=['date', 'locale_name'],
            how='left',
            suffixes=('', '_new')
        )

        # day_type 和 description 列处理
        for col in ['day_type_enc', 'description_enc']:
            df[col] = df[col].where(df[col].notnull(), df[f'{col}_new'])

        # 删除临时列
        df = df.drop(columns=['day_type_enc_new', 'description_enc_new', 'locale_name'])
        return df

    # 使用函数合并三类假期
    nat_holidays_agg = (
        nat_holidays.groupby("date").agg({
            "day_type_enc": lambda x: sum(x, []),
            "description_enc": lambda x: sum(x, []),
        }).reset_index()
    )
    df = df.merge(nat_holidays_agg, on="date", how="left")
    df = merge_holidays(df, reg_holidays, 'state')
    df = merge_holidays(df, loc_holidays, 'city')

    # 补充数据，对齐
    day_flag = np.where(df['date'].dt.dayofweek < 5, workday_num, endweek_num).tolist()
    df['description_enc'] = df['description_enc'].apply(lambda x: x if isinstance(x, list) else description_nan_num)
    df['day_type_enc'] = [[d] + (o if isinstance(o, list) else []) for d, o in zip(day_flag, df['day_type_enc'])]


    # 追加日期特征
    # 提取各个日期特征
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    df['weekday'] = df['date'].dt.weekday
    # 周期性特征（正弦和余弦变换）
    df['sin_month'] = np.sin(df['month'] * 2 * np.pi / 12)
    df['cos_month'] = np.cos(df['month'] * 2 * np.pi / 12)


    # 排序
    df = df.sort_values(["store_nbr", "family", "date"])
    # 再次计算每个列表的最大长度
    des_maxlen = max(df['description_enc'].apply(len))
    day_maxlen = max(df['day_type_enc'].apply(len))
    # 应用到df中的列
    df['description_enc'] = df['description_enc'].apply(
        lambda x: x + description_nan_num * (des_maxlen - len(x)) if len(x) < des_maxlen else x)
    df['day_type_enc'] = df['day_type_enc'].apply(
        lambda x: x + day_type_nan_num * (day_maxlen - len(x)) if len(x) < day_maxlen else x)

    # 其他feature补充
    df["salary_day"] = ((df["date"].dt.day == 15) |
                        (df["date"] == (df["date"] + pd.offsets.MonthEnd(0)))).astype(int)
    df["earthquake"] = ((df["date"] >= "2016-04-16") & (df["date"] < "2016-05-16")).astype(int)


    # ----------------------
    # TEST 文件处理
    # ----------------------
    df_test = test.merge(stores, on='store_nbr', how='left')
    df_test = df_test.merge(oil, on='date', how='left')
    # 填缺+改列名(防止重名)
    df_test["dcoilwtico"] = df_test["dcoilwtico"].interpolate(method="linear", limit_direction="both")
    df_test.rename(columns={"type": "stores_type"}, inplace=True)


    # 编码
    # 简单字符串编码
    df_test['city_enc'] = city_label.transform(df_test['city'])
    df_test['state_enc'] = state_label.transform(df_test['state'])
    df_test['family_enc'] = family_label.transform(df_test['family'])
    df_test['stores_type'] = stores_label.transform(df_test['stores_type'])

    # 使用函数合并三类假期
    df_test = df_test.merge(nat_holidays_agg, on="date", how="left")
    df_test = merge_holidays(df_test, reg_holidays, 'state')
    df_test = merge_holidays(df_test, loc_holidays, 'city')

    # 补充数据
    df_test['description_enc'] = df_test['description_enc'].apply(lambda x: x if isinstance(x, list) else description_nan_num)
    df_test['day_type_enc'] = [[d] + (o if isinstance(o, list) else []) for d, o in zip(day_flag, df_test['day_type_enc'])]

    # 追加日期特征
    # 提取各个日期特征
    df_test['year'] = df_test['date'].dt.year
    df_test['month'] = df_test['date'].dt.month
    df_test['day'] = df_test['date'].dt.day
    df_test['weekday'] = df_test['date'].dt.weekday
    # 周期性特征（正弦和余弦变换）
    df_test['sin_month'] = np.sin(df_test['month'] * 2 * np.pi / 12)
    df_test['cos_month'] = np.cos(df_test['month'] * 2 * np.pi / 12)

    # 排序
    df_test = df_test.sort_values(["store_nbr", "family", "date"])
    # 应用到df中的列
    df_test['description_enc'] = df_test['description_enc'].apply(
        lambda x: x + description_nan_num * (des_maxlen - len(x)) if len(x) < des_maxlen else x)
    df_test['day_type_enc'] = df_test['day_type_enc'].apply(
        lambda x: x + day_type_nan_num * (day_maxlen - len(x)) if len(x) < day_maxlen else x)

    # 其他feature补充
    df_test["salary_day"] = ((df_test["date"].dt.day == 15) |
                        (df_test["date"] == (df_test["date"] + pd.offsets.MonthEnd(0)))).astype(int)
    df_test["earthquake"] = ((df_test["date"] >= "2016-04-16") & (df_test["date"] < "2016-05-16")).astype(int)


    # 优化拆分方式，直接使用 NumPy 数组处理
    df_day = np.array(df['day_type_enc'].tolist())
    df_des = np.array(df['description_enc'].tolist())
    # 创建新的 DataFrame
    df_day_df = pd.DataFrame(df_day, columns=[f'day_type_{i}' for i in range(df_day.shape[1])])
    df_des_df = pd.DataFrame(df_des, columns=[f'description_{i}' for i in range(df_des.shape[1])])
    # 合并原 DataFrame 和拆分的列
    df = pd.concat([df, df_day_df, df_des_df], axis=1)

    # 对 df_test 进行相同的操作
    df_day_test = np.array(df_test['day_type_enc'].tolist())
    df_des_test = np.array(df_test['description_enc'].tolist())
    # 创建新的 DataFrame
    df_day_test_df = pd.DataFrame(df_day_test, columns=[f'day_type_{i}' for i in range(df_day_test.shape[1])])
    df_des_test_df = pd.DataFrame(df_des_test, columns=[f'description_{i}' for i in range(df_des_test.shape[1])])
    # 合并原 DataFrame 和拆分的列
    df_test = pd.concat([df_test, df_day_test_df, df_des_test_df], axis=1)


    # 删除不需要的列
    df.drop(columns=["family", "city", "state", "day_type_enc", "description_enc"], inplace=True)
    df_test.drop(columns=["family", "city", "state", "day_type_enc", "description_enc"], inplace=True)
    # 移动label到末尾
    col = df.pop("sales")
    df["sales"] = col
    # test添加 0 label
    df_test["sales"] = 0.0



    df.to_csv(root / 'preload_train.csv', index=False)
    df_test.to_csv(root / 'preload_test.csv', index=False)

    # return df, df_test

class MyDataset(Dataset):
    def __init__(self, data, seq_len, target_len):
        self.seq_len = seq_len
        self.target_len = target_len

        self.groups = [d.values.astype(np.float32) for _, d in data.groupby(["store_nbr", "family_enc"])]

        # 每个分组滑窗数量
        self.group_lens = [len(g) - seq_len - target_len + 1 for g in self.groups]
        self.cumsum_lens = np.cumsum([0] + self.group_lens)

    def __len__(self):
        return self.cumsum_lens[-1]

    def __getitem__(self, idx):
        # 找到idx属于哪个分组
        group_idx = np.searchsorted(self.cumsum_lens, idx, side='right') - 1
        idx_in_group = idx - self.cumsum_lens[group_idx]
        g = self.groups[group_idx]

        x = g[idx_in_group: idx_in_group + self.seq_len]
        y = g[idx_in_group + self.seq_len: idx_in_group + self.seq_len + self.target_len]
        return torch.tensor(x), torch.tensor(y)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))  # shape: (1, max_len, d_model)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return x

class TransformerModel(nn.Module):
    def __init__(self, src_fatures, tgt_fatures, d_model, nhead,
                 num_encoder_layers, num_decoder_layers, dim_hide,
                 dropout, max_seq_len):
        super().__init__()
        self.d_model = d_model
        self.src_input = nn.Linear(src_fatures, d_model)
        self.tgt_input = nn.Linear(tgt_fatures, d_model)
        self.encoder = PositionalEncoding(d_model, max_len=max_seq_len)
        self.decoder = PositionalEncoding(d_model, max_len=max_seq_len)

        self.transformer_encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                                       dim_feedforward=dim_hide, dropout=dropout, batch_first = True),
            num_layers=num_encoder_layers)

        self.transformer_decoder = nn.TransformerDecoder(
            nn.TransformerDecoderLayer(d_model=d_model, nhead=nhead,
                                       dim_feedforward=dim_hide, dropout=dropout, batch_first = True),
            num_layers=num_decoder_layers)

        self.mlp_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1),
        )

    def forward(self, src, tgt):
        """
        src: (batch, past_len, num_features)
        tgt: (batch, forecast, num_features) -> 训练时真实未来, 测试时可以传None
        return: (batch, forecast)
        """
        # ---------------- Encoder ----------------
        # 线性投影
        src_emb = self.src_input(src) # (batch, past_len, d_model)
        # 加位置编码
        src_emb = self.encoder(src_emb)                  # (batch, past_len, d_model)
        # Transformer Encoder
        memory = self.transformer_encoder(src_emb)            # (batch, past_len, d_model)

        # ---------------- Decoder ----------------
        tgt_emb = self.tgt_input(tgt)
        tgt_emb = self.decoder(tgt_emb)
        out_emb = self.transformer_decoder(tgt_emb, memory)

        out = self.mlp_head(out_emb)
        # out = 30 * torch.sigmoid(out)
        # if torch.isnan(out.any()):
        #     print("out NAN !!!")

        return out.squeeze(-1)

class RMSLELoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.mse = nn.MSELoss()

    def forward(self, y_pred, y_true):
        # y_pred = torch.clamp(y_pred, min=0, max=1e7)
        # y_true = torch.clamp(y_true, min=0, max=1e7)
        # if torch.any(torch.isnan(y_pred)):
        #     print("y_pred is NaN!")
        return torch.sqrt(
            self.mse(torch.log1p(y_pred), torch.log1p(y_true))
        )

class RMSLELossVal(nn.Module):
    def __init__(self):
        super().__init__()
        self.mse = nn.MSELoss()

    def forward(self, y_pred, y_true):
        # y_pred = torch.clamp(y_pred, min=0, max=1e7)
        # y_true = torch.clamp(y_true, min=0, max=1e7)
        # if torch.any(torch.isnan(y_pred)):
        #     print("y_pred is NaN!")
        return torch.sqrt(
            self.mse(y_pred, y_true)
        )

# ---- 参数 ----
lookback = 64  # 用过去 64 天的数据作为输入
forecast = 16  # 预测未来 16 天
batch_size = 256
lr = 1e-4
epochs = 10
src_fatures = 41 - 1
tgt_fatures = 41 - 2
d_model = 128
nhead = 4
num_encoder_layers = 2
num_decoder_layers = 2
dim_hide = 512
dropout = 0.2
max_seq_len = 300

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
save_path = "best_model.pth"

# model加载
model = TransformerModel(src_fatures, tgt_fatures, d_model, nhead,
                 num_encoder_layers, num_decoder_layers, dim_hide,
                 dropout, max_seq_len)
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = RMSLELoss()
criterion1 = nn.MSELoss()
criterion_val = RMSLELossVal()

# 训练
def evaluate(val_loader):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for src, tgt in val_loader:
            src, tgt = src.to(device, non_blocking=True), tgt.to(device, non_blocking=True)
            output = model(src[:, :, 1:], tgt[:, :, 1:-1])
            loss = criterion_val(output, tgt[:, :, -1])

            total_loss += loss.item() * src.size(0)

    return total_loss / len(val_loader.dataset)

def Train(train_loader, val_loader):
    best_loss = float("inf")
    scaler = torch.amp.GradScaler()

    num_training_steps = epochs * len(train_loader)
    # warmup 采用比例（例如 3%）
    num_warmup_steps = int(0.03 * num_training_steps)

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps,
        num_cycles=0.5
    )

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        train_pbar = tqdm(train_loader, position=0, leave=True)

        for src, tgt in train_pbar:
            src, tgt = src.to(device, non_blocking=True), tgt.to(device, non_blocking=True)

            optimizer.zero_grad()
            with torch.amp.autocast("cuda"):
                output = model(src[:, :, 1:], tgt[:, :, 1:-1])
                loss = criterion1(output, tgt[:, :, -1])

            scaler.scale(loss).backward()

            # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            # for name, param in model.named_parameters():
            #     if param.grad is not None:
            #         grad_norm = param.grad.norm().item()
            #         if grad_norm > 1e5:  # 阈值可以根据实际情况调整
            #             print(f"WARNING: Gradient explosion detected in {name}")

            scaler.step(optimizer)  # 先通过 scaler 处理梯度更新
            scaler.update()  # 更新 scaler
            scheduler.step()  # 更新学习率

            # if torch.isnan(loss):
            #     print("Loss is NaN!")
            #     return


            total_loss += loss.item() * src.size(0)
            train_pbar.set_description(f'Epoch [{epoch+1}/{epochs}]')
            train_pbar.set_postfix({'loss': f'{loss.item():.5f}'})

        train_loss = total_loss / len(train_loader.dataset)
        val_loss = evaluate(val_loader)
        print(f'Epoch [{epoch + 1}/{epochs}]: Train loss: {train_loss:.4f}, Valid loss: {val_loss:.4f}')

        if val_loss < best_loss:
            best_loss = val_loss
            torch.save(model.state_dict(), save_path)
            print(f'New best model saved with accuracy: {best_loss:.4f}')

def Train_(train_loader, val_loader):
    best_loss = float("inf")

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        train_pbar = tqdm(train_loader, position=0, leave=True)

        for src, tgt in train_pbar:
            src, tgt = src.to(device, non_blocking=True), tgt.to(device, non_blocking=True)

            optimizer.zero_grad()
            output = model(src[:, :, 1:], tgt[:, :, 1:-1])
            loss = criterion(output, tgt[:, :, -1])
            loss.backward()

            # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            # for name, param in model.named_parameters():
            #     if param.grad is not None:
            #         grad_norm = param.grad.norm().item()
            #         if grad_norm > 1e5:  # 阈值可以根据实际情况调整
            #             print(f"WARNING: Gradient explosion detected in {name}")

            optimizer.step()

            # if torch.isnan(loss):
            #     print("Loss is NaN!")
            #     return

            total_loss += loss.item() * src.size(0)
            train_pbar.set_description(f'Epoch [{epoch+1}/{epochs}]')
            train_pbar.set_postfix({'loss': f'{loss.item():.5f}'})

        train_loss = total_loss / len(train_loader.dataset)
        val_loss = evaluate(val_loader)
        print(f'Epoch [{epoch + 1}/{epochs}]: Train loss: {train_loss:.4f}, Valid loss: {val_loss:.4f}')

        if val_loss < best_loss:
            best_loss = val_loss
            torch.save(model.state_dict(), save_path)
            print(f'New best model saved with loss: {best_loss:.4f}')

def save_csv(test_loader):
    model = TransformerModel(src_fatures, tgt_fatures, d_model, nhead,
                             num_encoder_layers, num_decoder_layers, dim_hide,
                             dropout, max_seq_len)
    model.load_state_dict(torch.load(save_path))
    model = model.to(device)

    all_pred = []
    all_id = []
    model.eval()
    with torch.no_grad():
        for src, tgt in test_loader:
            src, tgt = src.to(device, non_blocking=True), tgt.to(device, non_blocking=True)
            outputs = model(src[:, :, 1:], tgt[:, :, 1:-1])
            outputs = torch.clamp(outputs, min=0)
            pred = outputs.cpu().numpy()
            ids = tgt[:, :, 0].cpu().numpy()
            all_pred.append(pred)
            all_id.append(ids)
        # 沿着批次维度拼接
        all_pred = np.concatenate(all_pred, axis=0)
        all_id = np.concatenate(all_id, axis=0)

    df_pred = pd.DataFrame({
        'id': all_id.reshape(-1),
        'sales': all_pred.reshape(-1),
    })
    print(df_pred.shape)


    # df_pred['sales'] = scaler.inverse_transform(df_pred['sales'])
    df_pred['sales'] = np.expm1(df_pred['sales'])
    df_pred['id'] = df_pred['id'].astype(int)

    df_pred = df_pred.sort_values(by='id').reset_index(drop=True)
    df_pred.to_csv('submission.csv', index=False)
    print("Save csv succeed")

def save_csv1(test_loader, df_test):
    model = TransformerModel(src_fatures, tgt_fatures, d_model, nhead,
                             num_encoder_layers, num_decoder_layers, dim_hide,
                             dropout, max_seq_len)
    model.load_state_dict(torch.load(save_path))
    model = model.to(device)

    total_loss = 0
    all_pred = []
    all_id = []
    model.eval()
    with torch.no_grad():
        for src, tgt in test_loader:
            src, tgt = src.to(device, non_blocking=True), tgt.to(device, non_blocking=True)
            outputs = model(src[:, :, 1:], tgt[:, :, 1:-1])
            outputs = torch.clamp(outputs, min=0)
            loss = criterion_val(outputs, tgt[:, :, -1])
            total_loss += loss.item() * src.size(0)

            pred = outputs.cpu().numpy()
            ids = tgt[:, :, 0].cpu().numpy()
            all_pred.append(pred)
            all_id.append(ids)
        all_id = np.concatenate(all_id, axis=0)
        all_pred = np.concatenate(all_pred, axis=0)
    print(f"loss: {total_loss / len(test_loader.dataset)}")
    df_pred = pd.DataFrame({
        'id': all_id.reshape(-1),
        'sales_pred': all_pred.reshape(-1),
    })
    print(df_pred.shape)

    # df_pred['sales'] = scaler.inverse_transform(df_pred['sales'])
    df_pred['sales_pred'] = np.expm1(df_pred['sales_pred'])
    df_test['sales'] = np.expm1(df_test['sales'])

    df_pred['id'] = df_pred['id'].astype(int)
    df_pred = df_pred.merge(df_test[["id", "sales"]], on="id", how='left')

    df_pred = df_pred.sort_values(by='id').reset_index(drop=True)
    df_pred.to_csv('submission1.csv', index=False)
    print("Save csv succeed")


if __name__ == '__main__':
    # 加载数据集
    # df, df_test = data_processing()
    # data_processing()
    df = pd.read_csv(root / 'preload_train.csv', parse_dates=["date"])
    df_test = pd.read_csv(root / 'preload_test.csv', parse_dates=["date"])

    # 拼接历史值
    df_test = pd.concat([df[df['date'] > df['date'].max() - pd.Timedelta(days=lookback)], df_test], axis=0, ignore_index=True)


    max_date = df["date"].max()
    valid_start = max_date - pd.Timedelta(days=forecast)

    # 切分训练集和验证集, 丢弃 date
    # 训练集: 所有日期 <= valid_start
    df_train = df[df["date"] <= valid_start]
    # 验证集: 所有日期 >= valid_start - pd.Timedelta(days=lookback)
    df_valid = df[df["date"] > valid_start - pd.Timedelta(days=lookback)]

    print(df_valid['date'].nunique(), df_test['date'].nunique())

    df_valid = df_valid.drop(columns=["date"])
    df_train = df_train.drop(columns=["date"])
    df_test.drop(columns=["date"], inplace=True)
    df_test = df_test[df_train.columns]

    # 取log
    df_train['sales'] = np.log1p(df_train['sales'])
    df_valid['sales'] = np.log1p(df_valid['sales'])
    df_test['sales'] = np.log1p(df_test['sales'])


    # # 对数值特征进行归一化
    # scaler = MinMaxScaler(feature_range=(0, 1))
    # df_train_ = df_train.copy()
    # df_valid_ = df_valid.copy()
    # df_test_ = df_test.copy()
    #
    # # 对特征部分进行归一化（排除'sales'列）
    # df_train_[df_train_.columns.difference(['sales'])] = scaler.fit_transform(df_train.drop(columns='sales'))
    # df_valid_[df_valid_.columns.difference(['sales'])] = scaler.transform(df_valid.drop(columns='sales'))
    # df_test_[df_test_.columns.difference(['sales'])] = scaler.transform(df_test.drop(columns='sales'))

    # 创建训练和验证数据集
    train_set = MyDataset(df_train , lookback, forecast)
    val_set = MyDataset(df_valid, lookback, forecast)
    test_set = MyDataset(df_test, lookback, forecast)


    # 推荐工作数
    rec_wk = 2
    print(f"recommended_workers: {rec_wk}")

    # 使用 DataLoader 加载训练、验证和测试集
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,
                              num_workers=rec_wk, pin_memory=True)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False,
                              num_workers=rec_wk, pin_memory=True)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False,
                              num_workers=rec_wk, pin_memory=True)

    # 开始训练并保存结果
    # Train(train_loader, val_loader)
    # save_csv(test_loader)
    # save_csv1(val_loader, df_valid)




In [ ]:
#code 2

import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
import math
from transformers import get_cosine_schedule_with_warmup
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler, RobustScaler
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils import clip_grad_norm_
from torch.optim.lr_scheduler import OneCycleLR
import numpy as np
from tqdm import tqdm
import re


pd.set_option('display.max_columns', None)
root = Path('data')

def data_processing():
    # 读数据
    train = pd.read_csv(root / "train.csv", parse_dates=["date"])
    test = pd.read_csv(root / "test.csv", parse_dates=["date"])
    stores = pd.read_csv(root / "stores.csv")
    oil = pd.read_csv(root / "oil.csv", parse_dates=["date"])
    transactions = pd.read_csv(root / "transactions.csv", parse_dates=["date"])
    holidays = pd.read_csv(root / "holidays_events.csv", parse_dates=["date"])
    sample = pd.read_csv(root / "sample_submission.csv")

    # 预处理train文件
    df = train.merge(stores, on='store_nbr', how='left')
    df = df.merge(oil, on='date', how='left')
    # 填缺+改列名(防止重名)
    df["dcoilwtico"] = df["dcoilwtico"].interpolate(method="linear", limit_direction="both")
    df.rename(columns={"type": "stores_type"}, inplace=True)


    # 预处理holidays文件 -> 处理文件字符串，改列名
    def clean_text(text):
        text = text.lower()
        text = re.sub(r"[^a-z ]", " ", text)  # 保留字母
        return text.strip().split()
    def clean_word(text):
        text_num = re.sub(r"[^1-9 +-]", " ", text).strip()
        return text_num

    holidays["desc_clean"] = holidays["description"].astype(str).map(clean_text)
    holidays["desc_num"] = holidays["description"].astype(str).map(clean_word)
    holidays['transferred'] = holidays['transferred'].astype(str)
    holidays['description'] = holidays.apply(lambda row: row["desc_clean"] + [row["desc_num"]], axis=1)
    holidays['type'] = holidays.apply(lambda row: [row["type"], row["transferred"]], axis=1)
    holidays.drop(columns=["desc_clean", "desc_num", "transferred"], inplace=True)
    holidays = holidays.rename(columns={"type": "day_type"})

    # 编码
    day_type_label = LabelEncoder()
    description_label = LabelEncoder()
    city_label = LabelEncoder()
    state_label = LabelEncoder()
    family_label = LabelEncoder()
    stores_label = LabelEncoder()
    # 简单字符串编码
    df['city_enc'] = city_label.fit_transform(df['city'])
    df['state_enc'] = state_label.fit_transform(df['state'])
    df['family_enc'] = family_label.fit_transform(df['family'])
    df['stores_type'] = stores_label.fit_transform(df['stores_type'])


    # 句子+复杂日期类型编码
    # 预编码
    day_type_all = [y for x in holidays['day_type'] for y in x] + ['Work Day', 'Endweek', 'nan']
    description_all = [y for x in holidays['description'] for y in x] + ['nan']
    day_type_label.fit_transform(day_type_all)
    description_label.fit_transform(description_all)
    # 编码holidays字符串
    holidays['day_type_enc'] = holidays['day_type'].apply(lambda x: day_type_label.transform(x))
    holidays['description_enc'] = holidays['description'].apply(lambda x: description_label.transform(x))

    # holidays处理
    # 按国家，州， 城市活动日期分类
    nat_holidays = holidays[holidays["locale"] == "National"]
    reg_holidays = holidays[holidays["locale"] == "Regional"]
    loc_holidays = holidays[holidays["locale"] == "Local"]

    # 求最大值
    des_maxlen = max(max(nat_holidays['description_enc'].apply(lambda x: len(x) if isinstance(x, np.ndarray) else [])),
                     max(reg_holidays['description_enc'].apply(lambda x: len(x) if isinstance(x, np.ndarray) else [])),
                     max(loc_holidays['description_enc'].apply(lambda x: len(x) if isinstance(x, np.ndarray) else [])))
    day_maxlen = max(max(nat_holidays['day_type_enc'].apply(lambda x: len(x) if isinstance(x, np.ndarray) else [])),
                     max(reg_holidays['day_type_enc'].apply(lambda x: len(x) if isinstance(x, np.ndarray) else [])),
                     max(loc_holidays['day_type_enc'].apply(lambda x: len(x) if isinstance(x, np.ndarray) else [])))

    # 补充编码提取
    description_nan_num = description_label.transform(['nan']).tolist()
    day_type_nan_num = day_type_label.transform(['nan']).tolist()
    workday_num = day_type_label.transform(['Work Day']).item()
    endweek_num = day_type_label.transform(['Endweek']).item()

    def process_holidays(df, des_maxlen, dty_maxlen, description_nan_num, day_type_nan_num):
        # 对需要处理的列执行统一的填充操作
        for col, max_len, pad_value in [('description_enc', des_maxlen, description_nan_num),
                                        ('day_type_enc', dty_maxlen, day_type_nan_num)]:
            df.loc[:, col] = df[col].apply(
                lambda x: x.tolist() + pad_value * (max_len - len(x)) if len(x) < max_len else x.tolist())
        return df

    # 对所有 DataFrame 执行相同的操作
    for holidays_df in [nat_holidays, reg_holidays, loc_holidays]:
        holidays_df = process_holidays(holidays_df, des_maxlen, day_maxlen, description_nan_num, day_type_nan_num)

    def merge_holidays(df, holidays_df, mask_col):
        """
        df: 主表
        holidays_df: 假期表
        on_col: 用来 merge 的列，一般是 'date'
        mask_col: 用来筛选行的列，比如 'state' 或 'city'
        """
        # 合并重复日期
        holidays_df = (
            holidays_df.groupby(["date", "locale_name"], as_index=False).agg({
                "day_type_enc": lambda x: sum(x, []),
                "description_enc": lambda x: sum(x, []),
            })
        )

        # merge 带新信息
        df = df.merge(
            holidays_df[['date', 'day_type_enc', 'description_enc', 'locale_name']],
            left_on=['date', mask_col],
            right_on=['date', 'locale_name'],
            how='left',
            suffixes=('', '_new')
        )

        # day_type 和 description 列处理
        for col in ['day_type_enc', 'description_enc']:
            df[col] = df[col].where(df[col].notnull(), df[f'{col}_new'])

        # 删除临时列
        df = df.drop(columns=['day_type_enc_new', 'description_enc_new', 'locale_name'])
        return df

    # 使用函数合并三类假期
    nat_holidays_agg = (
        nat_holidays.groupby("date").agg({
            "day_type_enc": lambda x: sum(x, []),
            "description_enc": lambda x: sum(x, []),
        }).reset_index()
    )
    df = df.merge(nat_holidays_agg, on="date", how="left")
    df = merge_holidays(df, reg_holidays, 'state')
    df = merge_holidays(df, loc_holidays, 'city')

    # 补充数据，对齐
    day_flag = np.where(df['date'].dt.dayofweek < 5, workday_num, endweek_num).tolist()
    df['description_enc'] = df['description_enc'].apply(lambda x: x if isinstance(x, list) else description_nan_num)
    df['day_type_enc'] = [[d] + (o if isinstance(o, list) else []) for d, o in zip(day_flag, df['day_type_enc'])]


    # 追加日期特征
    # 提取各个日期特征
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    df['weekday'] = df['date'].dt.weekday
    # 周期性特征（正弦和余弦变换）
    df['sin_month'] = np.sin(df['month'] * 2 * np.pi / 12)
    df['cos_month'] = np.cos(df['month'] * 2 * np.pi / 12)


    # 排序
    df = df.sort_values(["store_nbr", "family", "date"])
    # 再次计算每个列表的最大长度
    des_maxlen = max(df['description_enc'].apply(len))
    day_maxlen = max(df['day_type_enc'].apply(len))
    # 应用到df中的列
    df['description_enc'] = df['description_enc'].apply(
        lambda x: x + description_nan_num * (des_maxlen - len(x)) if len(x) < des_maxlen else x)
    df['day_type_enc'] = df['day_type_enc'].apply(
        lambda x: x + day_type_nan_num * (day_maxlen - len(x)) if len(x) < day_maxlen else x)

    # 其他feature补充
    df["salary_day"] = ((df["date"].dt.day == 15) |
                        (df["date"] == (df["date"] + pd.offsets.MonthEnd(0)))).astype(int)
    df["earthquake"] = ((df["date"] >= "2016-04-16") & (df["date"] < "2016-05-16")).astype(int)


    # ----------------------
    # TEST 文件处理
    # ----------------------
    df_test = test.merge(stores, on='store_nbr', how='left')
    df_test = df_test.merge(oil, on='date', how='left')
    # 填缺+改列名(防止重名)
    df_test["dcoilwtico"] = df_test["dcoilwtico"].interpolate(method="linear", limit_direction="both")
    df_test.rename(columns={"type": "stores_type"}, inplace=True)


    # 编码
    # 简单字符串编码
    df_test['city_enc'] = city_label.transform(df_test['city'])
    df_test['state_enc'] = state_label.transform(df_test['state'])
    df_test['family_enc'] = family_label.transform(df_test['family'])
    df_test['stores_type'] = stores_label.transform(df_test['stores_type'])

    # 使用函数合并三类假期
    df_test = df_test.merge(nat_holidays_agg, on="date", how="left")
    df_test = merge_holidays(df_test, reg_holidays, 'state')
    df_test = merge_holidays(df_test, loc_holidays, 'city')

    # 补充数据
    df_test['description_enc'] = df_test['description_enc'].apply(lambda x: x if isinstance(x, list) else description_nan_num)
    df_test['day_type_enc'] = [[d] + (o if isinstance(o, list) else []) for d, o in zip(day_flag, df_test['day_type_enc'])]

    # 追加日期特征
    # 提取各个日期特征
    df_test['year'] = df_test['date'].dt.year
    df_test['month'] = df_test['date'].dt.month
    df_test['day'] = df_test['date'].dt.day
    df_test['weekday'] = df_test['date'].dt.weekday
    # 周期性特征（正弦和余弦变换）
    df_test['sin_month'] = np.sin(df_test['month'] * 2 * np.pi / 12)
    df_test['cos_month'] = np.cos(df_test['month'] * 2 * np.pi / 12)

    # 排序
    df_test = df_test.sort_values(["store_nbr", "family", "date"])
    # 应用到df中的列
    df_test['description_enc'] = df_test['description_enc'].apply(
        lambda x: x + description_nan_num * (des_maxlen - len(x)) if len(x) < des_maxlen else x)
    df_test['day_type_enc'] = df_test['day_type_enc'].apply(
        lambda x: x + day_type_nan_num * (day_maxlen - len(x)) if len(x) < day_maxlen else x)

    # 其他feature补充
    df_test["salary_day"] = ((df_test["date"].dt.day == 15) |
                        (df_test["date"] == (df_test["date"] + pd.offsets.MonthEnd(0)))).astype(int)
    df_test["earthquake"] = ((df_test["date"] >= "2016-04-16") & (df_test["date"] < "2016-05-16")).astype(int)


    # 优化拆分方式，直接使用 NumPy 数组处理
    df_day = np.array(df['day_type_enc'].tolist())
    df_des = np.array(df['description_enc'].tolist())
    # 创建新的 DataFrame
    df_day_df = pd.DataFrame(df_day, columns=[f'day_type_{i}' for i in range(df_day.shape[1])])
    df_des_df = pd.DataFrame(df_des, columns=[f'description_{i}' for i in range(df_des.shape[1])])
    # 合并原 DataFrame 和拆分的列
    df = pd.concat([df, df_day_df, df_des_df], axis=1)

    # 对 df_test 进行相同的操作
    df_day_test = np.array(df_test['day_type_enc'].tolist())
    df_des_test = np.array(df_test['description_enc'].tolist())
    # 创建新的 DataFrame
    df_day_test_df = pd.DataFrame(df_day_test, columns=[f'day_type_{i}' for i in range(df_day_test.shape[1])])
    df_des_test_df = pd.DataFrame(df_des_test, columns=[f'description_{i}' for i in range(df_des_test.shape[1])])
    # 合并原 DataFrame 和拆分的列
    df_test = pd.concat([df_test, df_day_test_df, df_des_test_df], axis=1)


    # 删除不需要的列
    df.drop(columns=["family", "city", "state", "day_type_enc", "description_enc"], inplace=True)
    df_test.drop(columns=["family", "city", "state", "day_type_enc", "description_enc"], inplace=True)
    # 移动label到末尾
    col = df.pop("sales")
    df["sales"] = col
    # test添加 0 label
    df_test["sales"] = 0.0



    df.to_csv(root / 'preload_train.csv', index=False)
    df_test.to_csv(root / 'preload_test.csv', index=False)

    # return df, df_test

class MyDataset(Dataset):
    def __init__(self, data, seq_len, target_len):
        self.seq_len = seq_len
        self.target_len = target_len

        self.groups = [d.values.astype(np.float32) for _, d in data.groupby(["store_nbr", "family_enc"])]

        # 每个分组滑窗数量
        self.group_lens = [len(g) - seq_len - target_len + 1 for g in self.groups]
        self.cumsum_lens = np.cumsum([0] + self.group_lens)

    def __len__(self):
        return self.cumsum_lens[-1]

    def __getitem__(self, idx):
        # 找到idx属于哪个分组
        group_idx = np.searchsorted(self.cumsum_lens, idx, side='right') - 1
        idx_in_group = idx - self.cumsum_lens[group_idx]
        g = self.groups[group_idx]

        x = g[idx_in_group: idx_in_group + self.seq_len]
        y = g[idx_in_group + self.seq_len: idx_in_group + self.seq_len + self.target_len]
        return torch.tensor(x), torch.tensor(y)

class MyDataset_(Dataset):
    def __init__(self, data, seq_len, target_len):
        self.seq_len = seq_len
        self.target_len = target_len

        # 先按天分组（保证每组就是一天的数据，顺序必须按照日期排序）
        self.days = [d.drop(columns=["date"]).values.astype(np.float32) for _, d in data.groupby("date")]

        # 每个样本就是连续 (seq_len + target_len) 天
        self.num_samples = len(self.days) - seq_len - target_len + 1

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        # 连续天片段
        window = self.days[idx: idx + self.seq_len + self.target_len]

        # 拼接成 tensor
        window = np.stack(window)  # (seq_len+target_len, rows_per_day, features)

        x = window[:self.seq_len]       # 历史
        y = window[self.seq_len:]       # 未来
        return torch.tensor(x), torch.tensor(y)

def generate_square_subsequent_mask(sz, device):
    # causal mask: shape (sz, sz), True/False for masked positions in newer versions
    mask = torch.triu(torch.full((sz, sz), float('-inf')), diagonal=1).to(device)
    return mask  # pass directly to transformer (expects float mask)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        # x: (B, L, D)
        x = x + self.pe[:, :x.size(1), :].to(x.dtype)
        return x

class TransformerForecast(nn.Module):
    def __init__(self, src_features, tgt_features, d_model=128, nhead=8,
                 num_encoder_layers=3, num_decoder_layers=3, dim_feedforward=512,
                 dropout=0.1, max_seq_len=512):
        super().__init__()
        self.d_model = d_model
        self.src_input = nn.Linear(src_features, d_model)
        self.tgt_input = nn.Linear(tgt_features, d_model)

        self.pos_enc = PositionalEncoding(d_model, max_len=max_seq_len)

        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                                                   dim_feedforward=dim_feedforward,
                                                   dropout=dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer,
                                                         num_layers=num_encoder_layers)

        decoder_layer = nn.TransformerDecoderLayer(d_model=d_model, nhead=nhead,
                                                   dim_feedforward=dim_feedforward,
                                                   dropout=dropout, batch_first=True)
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer,
                                                         num_layers=num_decoder_layers)

        self.input_layernorm = nn.LayerNorm(d_model)
        self.output_layernorm = nn.LayerNorm(d_model)

        self.mlp_head = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, 1),
        )

    def forward(self, src, tgt, tgt_mask=None):
        # src: (B, past_len, src_features)
        # tgt: (B, fut_len, tgt_features)  -- during training provide shifted inputs
        device = src.device
        src_emb = self.src_input(src) * math.sqrt(self.d_model)
        src_emb = self.pos_enc(src_emb)
        src_emb = self.input_layernorm(src_emb)
        memory = self.transformer_encoder(src_emb)  # (B, past_len, D)

        tgt_emb = self.tgt_input(tgt) * math.sqrt(self.d_model)
        tgt_emb = self.pos_enc(tgt_emb)
        tgt_emb = self.input_layernorm(tgt_emb)
        if tgt_mask is None:
            fut_len = tgt_emb.size(1)
            tgt_mask = generate_square_subsequent_mask(fut_len, device)  # (L,L)
        out_emb = self.transformer_decoder(tgt_emb, memory, tgt_mask=tgt_mask)  # (B, fut_len, D)
        out_emb = self.output_layernorm(out_emb)
        out = self.mlp_head(out_emb).squeeze(-1)  # (B, fut_len)
        return out

class RMSLELoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.mse = nn.MSELoss()

    def forward(self, y_pred, y_true):
        # y_pred = torch.clamp(y_pred, min=0, max=1e7)
        # y_true = torch.clamp(y_true, min=0, max=1e7)
        # if torch.any(torch.isnan(y_pred)):
        #     print("y_pred is NaN!")
        return torch.sqrt(
            self.mse(torch.log1p(y_pred), torch.log1p(y_true))
        )

class RMSLELossVal(nn.Module):
    def __init__(self):
        super().__init__()
        self.mse = nn.MSELoss()

    def forward(self, y_pred, y_true):
        y_pred = torch.clamp(y_pred, min=0)
        # y_true = torch.clamp(y_true, min=0, max=1e7)
        # if torch.any(torch.isnan(y_pred)):
        #     print("y_pred is NaN!")
        return torch.sqrt(
            self.mse(y_pred, y_true)
        )

# ---- 参数 ----
lookback = 180  # 用过去 64 天的数据作为输入
forecast = 16  # 预测未来 16 天
batch_size = 512
lr = 1e-4
epochs = 30
src_fatures = 41 - 1
tgt_fatures = 41 - 2 + 1
d_model = 256
nhead = 4
num_encoder_layers = 2
num_decoder_layers = 2
dim_hide = 512
dropout = 0.3
max_seq_len = 300

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
save_path = "best_model.pth"

# model加载
# model = TransformerModel(src_fatures, tgt_fatures, d_model, nhead,
#                  num_encoder_layers, num_decoder_layers, dim_hide,
#                  dropout, max_seq_len)
model = TransformerForecast(src_fatures, tgt_fatures, d_model, nhead,
                 num_encoder_layers, num_decoder_layers, dim_hide,
                 dropout, max_seq_len)
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay = 0.01)
criterion = RMSLELoss()
criterion1 = nn.MSELoss()
criterion_val = RMSLELossVal()

# 训练
def evaluate(val_loader):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for src, tgt in val_loader:
            src, tgt = src.to(device, non_blocking=True), tgt.to(device, non_blocking=True)
            # start token: 历史最后一天的 label
            start_token = src[:, -1:, -1:]  # shape = (B, 1, 1)
            # 未来标签右移一位（去掉最后一天）
            shifted_labels = tgt[:, :-1, -1:]  # shape = (B, fut_len-1, 1)
            # 拼接形成 decoder 的标签输入
            tgt_label_input = torch.cat([start_token, shifted_labels], dim=1)  # (B, fut_len, 1)
            # 未来特征（id + 特征列）
            future_features = tgt[:, :, 1:-1]  # shape = (B, fut_len, id+features)
            tgt_input = torch.cat([future_features, tgt_label_input], dim=-1)  # shape = (B, fut_len, id+features+label)

            output = model(src[:, :, 1:], tgt_input)
            loss = criterion_val(output, tgt[:, :, -1])

            total_loss += loss.item() * src.size(0)

    return total_loss / len(val_loader.dataset)

def Train(train_loader, val_loader):
    best_loss = float("inf")
    early_stop = 0
    scaler = torch.amp.GradScaler()

    num_training_steps = epochs * len(train_loader)
    # warmup 采用比例（例如 3%）
    num_warmup_steps = int(0.1 * num_training_steps)

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps,
        num_cycles=0.5
    )

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0


        train_pbar = tqdm(train_loader, position=0, leave=True)

        for src, tgt in train_pbar:
            src, tgt = src.to(device, non_blocking=True), tgt.to(device, non_blocking=True)

            optimizer.zero_grad()
            with torch.amp.autocast("cuda"):
                # start token: 历史最后一天的 label
                start_token = src[:, -1:, -1:]  # shape = (B, 1, 1)
                # 未来标签右移一位（去掉最后一天）
                shifted_labels = tgt[:, :-1, -1:]  # shape = (B, fut_len-1, 1)
                # 拼接形成 decoder 的标签输入
                tgt_label_input = torch.cat([start_token, shifted_labels], dim=1)  # (B, fut_len, 1)
                # 未来特征（id + 特征列）
                future_features = tgt[:, :, 1:-1]  # shape = (B, fut_len, id+features)
                tgt_input = torch.cat([future_features, tgt_label_input], dim=-1)  # shape = (B, fut_len, id+features+label)

                output = model(src[:, :, 1:], tgt_input)
                loss = criterion1(output, tgt[:, :, -1])

            scaler.scale(loss).backward()

            # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            # for name, param in model.named_parameters():
            #     if param.grad is not None:
            #         grad_norm = param.grad.norm().item()
            #         if grad_norm > 1e5:  # 阈值可以根据实际情况调整
            #             print(f"WARNING: Gradient explosion detected in {name}")

            scaler.step(optimizer)  # 先通过 scaler 处理梯度更新
            scaler.update()  # 更新 scaler
            scheduler.step()  # 更新学习率

            # if torch.isnan(loss):
            #     print("Loss is NaN!")
            #     return


            total_loss += loss.item() * src.size(0)
            train_pbar.set_description(f'Epoch [{epoch+1}/{epochs}]')
            train_pbar.set_postfix({'loss': f'{loss.item():.5f}'})

        train_loss = total_loss / len(train_loader.dataset)
        val_loss = evaluate(val_loader)
        print(f'Epoch [{epoch + 1}/{epochs}]: Train loss: {train_loss:.4f}, Valid loss: {val_loss:.4f}')

        if val_loss < best_loss:
            best_loss = val_loss
            early_stop = 0
            torch.save(model.state_dict(), save_path)
            print(f'New best model saved with loss: {best_loss:.4f}')
        else:
            early_stop += 1
            if early_stop >= 5:
                break
    print(f"best loss: {best_loss}")

def Train_(train_loader, val_loader):
    best_loss = float("inf")

    # 创建 OneCycleLR 调度器
    scheduler = OneCycleLR(
        optimizer,
        max_lr=1e-3,  # 最大学习率（峰值）
        epochs=epochs,  # 总训练轮数
        steps_per_epoch=len(train_loader),  # 每个 epoch 的 batch 数
        pct_start=0.3,  # 前 30% 的训练时间用于升高 lr
        anneal_strategy="cos",  # 后面余弦下降
        div_factor=25.0  # 初始 lr = max_lr / div_factor
    )

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        train_pbar = tqdm(train_loader, position=0, leave=True)

        for src, tgt in train_pbar:
            src, tgt = src.to(device, non_blocking=True), tgt.to(device, non_blocking=True)

            optimizer.zero_grad()
            output = model(src[:, :, 1:], tgt[:, :, 1:-1])
            loss = criterion(output, tgt[:, :, -1])
            loss.backward()

            # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            # for name, param in model.named_parameters():
            #     if param.grad is not None:
            #         grad_norm = param.grad.norm().item()
            #         if grad_norm > 1e5:  # 阈值可以根据实际情况调整
            #             print(f"WARNING: Gradient explosion detected in {name}")

            optimizer.step()
            # scheduler.step()

            if torch.isnan(loss):
                print("Loss is NaN!")
                return

            total_loss += loss.item() * src.size(0)
            train_pbar.set_description(f'Epoch [{epoch+1}/{epochs}]')
            train_pbar.set_postfix({'loss': f'{loss.item():.5f}'})

        train_loss = total_loss / len(train_loader.dataset)
        val_loss = evaluate(val_loader)
        print(f'Epoch [{epoch + 1}/{epochs}]: Train loss: {train_loss:.4f}, Valid loss: {val_loss:.4f}')

        if val_loss < best_loss:
            best_loss = val_loss
            torch.save(model.state_dict(), save_path)
            print(f'New best model saved with loss: {best_loss:.4f}')

def save_csv(test_loader):
    model = TransformerForecast(src_fatures, tgt_fatures, d_model, nhead,
                                num_encoder_layers, num_decoder_layers, dim_hide,
                                dropout, max_seq_len)
    model.load_state_dict(torch.load("best_model.pth"))
    model = model.to(device)

    all_pred = []
    all_id = []
    model.eval()
    with torch.no_grad():
        for src, tgt in test_loader:
            src = src.to(device, non_blocking=True)  # (B, past_len, id+feat+label)
            tgt = tgt.to(device, non_blocking=True)  # (B, fut_len, id+feat+label_or_no_label)

            B, fut_len, _ = tgt.shape
            # 保存 id（用于输出）
            ids = tgt[:, :, 0].cpu().numpy()
            all_id.append(ids)
            # 历史最后一天的 label 作为 start token (B,1,1)
            start_token = src[:, -1:, -1:].to(device)  # label 在最后一列
            # future_features: 去掉 id(0) 和 label(最后一列)
            future_features = tgt[:, :, 1:-1]  # shape (B, fut_len, feat_dim)
            # 初始化 decoder_input：第一步用 future_features 的第一天 + start_token
            # future_features[:, :1, :] -> (B,1,feat_dim)
            decoder_input = torch.cat([future_features[:, :1, :], start_token], dim=-1)  # (B,1,feat_dim+1)

            # 自回归生成
            preds_batch = []  # 存每一步 (B,1)
            for t in range(fut_len):
                out = model(src[:, :, 1:], decoder_input)  # 把 id 去掉传入模型， out shape (B, current_len)
                # 取最后一步预测，形状 (B,1)
                next_val = out[:, -1:].to(device)  # (B,1)

                # 保存为 (B,1) 以便最终拼接成 (B, fut_len)
                preds_batch.append(next_val.cpu())

                # 如果还有下步，把 next_feat (B,1,feat_dim) 与 next_val 合并，作为下步 decoder 输入
                if t + 1 < fut_len:
                    next_feat = future_features[:, t + 1:t + 2, :]  # (B,1,feat_dim)
                    # next_val 是 (B,1) -> 先扩成 (B,1,1) 再按最后维度拼接
                    next_val_exp = next_val.unsqueeze(-1)  # (B,1,1)
                    next_step = torch.cat([next_feat, next_val_exp], dim=-1)  # (B,1,feat_dim+1)
                    decoder_input = torch.cat([decoder_input, next_step], dim=1)  # (B, L+1, feat_dim+1)

            # preds_batch 中每项是 (B,1) 的 cpu-tensor，拼成 (B, fut_len)
            preds = torch.cat(preds_batch, dim=1).clamp(min = 0.0).numpy()  # (B, fut_len)
            all_pred.append(preds)

        # 沿 batch 维拼接
        all_pred = np.concatenate(all_pred, axis=0)  # (N_samples, fut_len)
        all_id = np.concatenate(all_id, axis=0)  # (N_samples, fut_len) 或 (N_samples, ?)



    df_pred = pd.DataFrame({
        'id': all_id.reshape(-1),
        'sales': all_pred.reshape(-1),
    })
    print(df_pred.shape)


    # df_pred['sales'] = scaler.inverse_transform(df_pred['sales'])
    df_pred['sales'] = np.expm1(df_pred['sales'])
    df_pred['id'] = df_pred['id'].astype(int)

    df_pred = df_pred.sort_values(by='id').reset_index(drop=True)
    df_pred.to_csv('submission.csv', index=False)
    print("Save csv succeed")

def save_csv1(test_loader, df_test):
    model = TransformerForecast(src_fatures, tgt_fatures, d_model, nhead,
                                num_encoder_layers, num_decoder_layers, dim_hide,
                                dropout, max_seq_len)
    model.load_state_dict(torch.load(save_path))
    model = model.to(device)

    total_loss = 0
    all_pred = []
    all_id = []
    model.eval()
    with torch.no_grad():
        for src, tgt in test_loader:
            src = src.to(device, non_blocking=True)  # (B, past_len, id+feat+label)
            tgt = tgt.to(device, non_blocking=True)  # (B, fut_len, id+feat+label_or_no_label)

            # start token: 历史最后一天的 label
            start_token = src[:, -1:, -1:]  # shape = (B, 1, 1)
            # 未来标签右移一位（去掉最后一天）
            shifted_labels = tgt[:, :-1, -1:]  # shape = (B, fut_len-1, 1)
            # 拼接形成 decoder 的标签输入
            tgt_label_input = torch.cat([start_token, shifted_labels], dim=1)  # (B, fut_len, 1)
            # 未来特征（id + 特征列）
            future_features = tgt[:, :, 1:-1]  # shape = (B, fut_len, id+features)
            tgt_input = torch.cat([future_features, tgt_label_input], dim=-1)  # shape = (B, fut_len, id+features+label)

            output = model(src[:, :, 1:], tgt_input)
            loss = criterion_val(output, tgt[:, :, -1])

            total_loss += loss.item() * src.size(0)

            B, fut_len, _ = tgt.shape
            # 保存 id（用于输出）
            ids = tgt[:, :, 0].cpu().numpy()
            all_id.append(ids)
            # 历史最后一天的 label 作为 start token (B,1,1)
            start_token = src[:, -1:, -1:].to(device)  # label 在最后一列
            # future_features: 去掉 id(0) 和 label(最后一列)
            future_features = tgt[:, :, 1:-1]  # shape (B, fut_len, feat_dim)
            # 初始化 decoder_input：第一步用 future_features 的第一天 + start_token
            # future_features[:, :1, :] -> (B,1,feat_dim)
            decoder_input = torch.cat([future_features[:, :1, :], start_token], dim=-1)  # (B,1,feat_dim+1)

            # 自回归生成
            preds_batch = []  # 存每一步 (B,1)
            for t in range(fut_len):
                out = model(src[:, :, 1:], decoder_input)  # 把 id 去掉传入模型， out shape (B, current_len)
                # 取最后一步预测，形状 (B,1)
                next_val = out[:, -1:].to(device)  # (B,1)

                # 保存为 (B,1) 以便最终拼接成 (B, fut_len)
                preds_batch.append(next_val.cpu())

                # 如果还有下步，把 next_feat (B,1,feat_dim) 与 next_val 合并，作为下步 decoder 输入
                if t + 1 < fut_len:
                    next_feat = future_features[:, t + 1:t + 2, :]  # (B,1,feat_dim)
                    # next_val 是 (B,1) -> 先扩成 (B,1,1) 再按最后维度拼接
                    next_val_exp = next_val.unsqueeze(-1)  # (B,1,1)
                    next_step = torch.cat([next_feat, next_val_exp], dim=-1)  # (B,1,feat_dim+1)
                    decoder_input = torch.cat([decoder_input, next_step], dim=1)  # (B, L+1, feat_dim+1)

            # preds_batch 中每项是 (B,1) 的 cpu-tensor，拼成 (B, fut_len)
            preds = torch.cat(preds_batch, dim=1).numpy()  # (B, fut_len)
            all_pred.append(preds)

        # 沿 batch 维拼接
        all_pred = np.concatenate(all_pred, axis=0)  # (N_samples, fut_len)
        all_id = np.concatenate(all_id, axis=0)  # (N_samples, fut_len) 或 (N_samples, ?)

    print(f"loss: {total_loss / len(test_loader.dataset)}")
    df_pred = pd.DataFrame({
        'id': all_id.reshape(-1),
        'sales_pred': all_pred.reshape(-1),
    })
    print(df_pred.shape)

    # df_pred['sales'] = scaler.inverse_transform(df_pred['sales'])
    df_pred['sales_pred'] = np.expm1(df_pred['sales_pred'])
    df_test['sales'] = np.expm1(df_test['sales'])

    df_pred['id'] = df_pred['id'].astype(int)
    df_pred = df_pred.merge(df_test[["id", "sales"]], on="id", how='left')

    df_pred = df_pred.sort_values(by='id').reset_index(drop=True)
    df_pred.to_csv('submission1.csv', index=False)
    print("Save csv succeed")

if __name__ == '__main__':
    # 加载数据集
    # df, df_test = data_processing()
    # data_processing()
    df = pd.read_csv(root / 'preload_train.csv', parse_dates=["date"])
    df_test = pd.read_csv(root / 'preload_test.csv', parse_dates=["date"])

    print(df_test['date'].nunique())

    # 拼接历史值
    df_test = pd.concat([df[df['date'] > df['date'].max() - pd.Timedelta(days=lookback)], df_test], axis=0,
                        ignore_index=True)

    max_date = df["date"].max()
    valid_start = max_date - pd.Timedelta(days=forecast)

    # 切分训练集和验证集, 丢弃 date
    # 训练集: 所有日期 <= valid_start
    df_train = df[df["date"] <= valid_start]
    # 验证集: 所有日期 >= valid_start - pd.Timedelta(days=lookback-1)
    df_valid = df[df["date"] > valid_start - pd.Timedelta(days=lookback)]

    print(df_valid['date'].nunique(), df_test['date'].nunique())

    df_valid = df_valid.drop(columns=["date"])
    df_train = df_train.drop(columns=["date"])
    df_test.drop(columns=["date"], inplace=True)
    df_test = df_test[df_train.columns]


    # 取log
    df_train.loc[:, 'sales'] = np.log1p(df_train['sales'])
    df_valid.loc[:, 'sales'] = np.log1p(df_valid['sales'])
    df_test['sales'] = np.log1p(df_test['sales'])

    print(len(df_test))

    # # 对数值特征进行归一化
    # scaler = MinMaxScaler(feature_range=(0, 1))
    # df_train_ = df_train.copy()
    # df_valid_ = df_valid.copy()
    # df_test_ = df_test.copy()
    #
    # # 对特征部分进行归一化（排除'sales'列）
    # df_train_[df_train_.columns.difference(['sales'])] = scaler.fit_transform(df_train.drop(columns='sales'))
    # df_valid_[df_valid_.columns.difference(['sales'])] = scaler.transform(df_valid.drop(columns='sales'))
    # df_test_[df_test_.columns.difference(['sales'])] = scaler.transform(df_test.drop(columns='sales'))

    # 创建训练和验证数据集
    train_set = MyDataset(df_train , lookback, forecast)
    val_set = MyDataset(df_valid, lookback, forecast)
    test_set = MyDataset(df_test, lookback, forecast)


    # 推荐工作数
    rec_wk = 2
    print(f"recommended_workers: {rec_wk}")


    # 使用 DataLoader 加载训练、验证和测试集
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,
                              num_workers=rec_wk, pin_memory=True)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False,
                              num_workers=rec_wk, pin_memory=True)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False,
                              num_workers=rec_wk, pin_memory=True)


    # 开始训练并保存结果
    # Train(train_loader, val_loader)
    # save_csv(test_loader)
    # save_csv1(val_loader, df_valid)


